# Modellvergleich: spätere Kursabmeldung

Wir vergleichen eine einfache Aktivitätsregel mit einem Modell, das mehrere bis Tag 14 bekannte Merkmale verwendet. Die Daten werden nach `id_student` aufgeteilt: Kurseinträge derselben Person bleiben vollständig im Training oder vollständig im Test. Die Testdaten dienen erst am Ende zur Bewertung.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Pfad zur gespeicherten Modellbasis.
projektordner = Path(
    r"C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt"
)
modelldatei = projektordner / "data" / "processed" / "oulad_tag14_modellbasis_v2.csv"
daten = pd.read_csv(modelldatei)

# Ganze Personen statt einzelner Zeilen aufteilen.
aufteilung = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)
train_index, test_index = next(
    aufteilung.split(daten, groups=daten["id_student"])
)

train = daten.iloc[train_index].copy()
test = daten.iloc[test_index].copy()

print("Training:", len(train), "Kurseinträge")
print("Test:", len(test), "Kurseinträge")
print(
    "Personen in beiden Gruppen:",
    len(set(train["id_student"]) & set(test["id_student"]))
)
print("Abbruchquote Training:", round(train["spaeter_abgemeldet"].mean() * 100, 1), "%")
print("Abbruchquote Test:", round(test["spaeter_abgemeldet"].mean() * 100, 1), "%")

Training: 22371 Kurseinträge
Test: 5590 Kurseinträge
Personen in beiden Gruppen: 0
Abbruchquote Training: 20.1 %
Abbruchquote Test: 19.3 %


## Entwicklung und Validierung

Der Testsatz bleibt bis zur abschließenden Bewertung ungenutzt. Innerhalb der Trainingsdaten teilen wir Personen erneut in Entwicklung und Validierung auf. Beide Verfahren dürfen gleich viele Hinweise erzeugen: zunächst 20 % der Kurseinträge in der Validierungsgruppe. Wir vergleichen, wie viele spätere Abmeldungen darunter sind.

In [2]:
# Personen innerhalb der Trainingsdaten nochmals aufteilen.
# Der bisherige Testsatz bleibt dabei unberührt.
innere_aufteilung = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=43
)
entwick_index, valid_index = next(
    innere_aufteilung.split(train, groups=train["id_student"])
)

entwick = train.iloc[entwick_index].copy()
valid = train.iloc[valid_index].copy()

# Die Klicks aus beiden Zeitfenstern zusammenzählen.
valid["klicks_gesamt"] = (
    valid["klicks_tag_0_bis_6"]
    + valid["klicks_tag_7_bis_14"]
)

# Beispielhafte Kapazität: Hinweise für rund 20 % der Gruppe.
kontaktanteil = 0.20
anzahl_hinweise = round(len(valid) * kontaktanteil)

# Einfache Regel: die Kurseinträge mit den wenigsten Klicks auswählen.
hinweise_regel = valid.nsmallest(
    anzahl_hinweise, "klicks_gesamt"
)

# Rückblickend prüfen, wie viele dieser Hinweise zutrafen.
treffer = hinweise_regel["spaeter_abgemeldet"].sum()
alle_abmeldungen = valid["spaeter_abgemeldet"].sum()

print("Entwicklung:", len(entwick), "Kurseinträge")
print("Validierung:", len(valid), "Kurseinträge")
print(
    "Personen in beiden Gruppen:",
    len(set(entwick["id_student"]) & set(valid["id_student"]))
)
print("Hinweise der einfachen Regel:", anzahl_hinweise)
print("Davon spätere Abmeldungen:", treffer)
print("Trefferquote der Hinweise:", round(treffer / anzahl_hinweise * 100, 1), "%")
print(
    "Anteil aller späteren Abmeldungen gefunden:",
    round(treffer / alle_abmeldungen * 100, 1), "%"
)

Entwicklung: 16800 Kurseinträge
Validierung: 5571 Kurseinträge
Personen in beiden Gruppen: 0
Hinweise der einfachen Regel: 1114
Davon spätere Abmeldungen: 286
Trefferquote der Hinweise: 25.7 %
Anteil aller späteren Abmeldungen gefunden: 25.8 %


## Erstes Mehrvariablenmodell

Eine logistische Regression kombiniert mehrere bis Tag 14 bekannte Merkmale. Numerische Werte werden skaliert, Kurskennungen als Kategorien verarbeitet. Diese Verarbeitung wird ausschließlich an den Entwicklungsdaten angepasst. Für den Vergleich wählen wir in der Validierung dieselbe Anzahl von Hinweisen wie bei der einfachen Regel.

In [3]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)

# Stark streuende Klickzahlen erst mit log(1 + Klicks)
# verkleinern und danach skalieren.
klickspalten = ["klicks_tag_0_bis_6", "klicks_tag_7_bis_14"]
klickverarbeitung = Pipeline([
    ("log", FunctionTransformer(np.log1p)),
    ("skalierung", StandardScaler()),
])

# Die übrigen Zahlen ebenfalls skalieren.
zahlenspalten = ["anmeldetag",
    "aktive_tage_0_bis_6",
    "aktive_tage_7_bis_14",]

# Modul und Kursdurchlauf sind Kategorien, keine Zahlen mit Abständen.
kategoriespalten = ["code_module", "code_presentation"]

vorverarbeitung = ColumnTransformer([
    ("klicks", klickverarbeitung, klickspalten),
    ("zahlen", StandardScaler(), zahlenspalten),
    ("kategorien", OneHotEncoder(handle_unknown="ignore"), kategoriespalten),
])

modell = Pipeline([
    ("vorverarbeitung", vorverarbeitung),
    ("klassifikation", LogisticRegression(max_iter=1000)),
])

# id_student und die Zielvariable gehören NICHT zu den Eingaben.
merkmale = klickspalten + zahlenspalten + kategoriespalten

modell.fit(
    entwick[merkmale],
    entwick["spaeter_abgemeldet"]
)

print("Modell trainiert.")

Modell trainiert.


In [4]:
# Modellwerte für die Validierungsgruppe berechnen.
# Spalte 1 gehört zur Klasse "spätere Abmeldung = True".
modellwerte = modell.predict_proba(valid[merkmale])[:, 1]

# Nur für die Auswertung eine Kopie mit Modellwert anlegen.
valid_mit_wert = valid.copy()
valid_mit_wert["modellwert"] = modellwerte

# Genau so viele Fälle auswählen wie bei der einfachen Klickregel.
hinweise_modell = valid_mit_wert.nlargest(
    anzahl_hinweise, "modellwert"
)

modell_treffer = hinweise_modell["spaeter_abgemeldet"].sum()
alle_abmeldungen = valid["spaeter_abgemeldet"].sum()

print("Hinweise:", len(hinweise_modell))
print("Davon spätere Abmeldungen:", modell_treffer)
print(
    "Trefferquote der Hinweise:",
    round(modell_treffer / anzahl_hinweise * 100, 1),
    "%"
)
print(
    "Anteil aller späteren Abmeldungen gefunden:",
    round(modell_treffer / alle_abmeldungen * 100, 1),
    "%"
)
print("Treffer der einfachen Regel bei gleicher Hinweiszahl:", treffer)

Hinweise: 1114
Davon spätere Abmeldungen: 389
Trefferquote der Hinweise: 34.9 %
Anteil aller späteren Abmeldungen gefunden: 35.1 %
Treffer der einfachen Regel bei gleicher Hinweiszahl: 286


In [5]:
from sklearn.metrics import average_precision_score

# Für die einfache Regel bedeuten weniger Klicks höhere Priorität.
# Das Minuszeichen dreht deshalb die Sortierrichtung um.
regelwerte = -valid["klicks_gesamt"]

# Average Precision bewertet die gesamte Rangfolge.
ap_regel = average_precision_score(
    valid["spaeter_abgemeldet"], regelwerte
)
ap_modell = average_precision_score(
    valid["spaeter_abgemeldet"], modellwerte
)

print("Average Precision – Klickregel:", round(ap_regel, 3))
print("Average Precision – Modell:", round(ap_modell, 3))
print(
    "Abbruchanteil der Validierungsgruppe:",
    round(valid["spaeter_abgemeldet"].mean(), 3)
)

Average Precision – Klickregel: 0.239
Average Precision – Modell: 0.33
Abbruchanteil der Validierungsgruppe: 0.199


### Priorität der Bewertung

Das Projekt priorisiert einen hohen Recall: Möglichst viele spätere Kursabmeldungen sollen frühzeitig als mögliche Unterstützungsfälle erkannt werden. Zusätzliche Hinweise an Personen, die sich später nicht abmelden, sind eher akzeptabel als übersehene Abmeldungen. Wir berichten deshalb Recall, Trefferquote und erforderliche Anzahl von Hinweisen gemeinsam und wählen die Hinweisgrenze anhand der Validierungsdaten.

In [6]:
# Kurseinträge nach Modellwert sortieren: höchster Wert zuerst.
rangfolge = (
    valid.assign(modellwert=modellwerte)
    .sort_values("modellwert", ascending=False)
    .reset_index(drop=True)
)

alle_abmeldungen = rangfolge["spaeter_abgemeldet"].sum()
zeilen = []

for anteil in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    # So viele Kurseinträge würden einen Hinweis erhalten.
    anzahl = round(len(rangfolge) * anteil)
    ausgewaehlt = rangfolge.iloc[:anzahl]

    treffer = ausgewaehlt["spaeter_abgemeldet"].sum()

    zeilen.append({
        "Hinweisanteil": f"{anteil:.0%}",
        "Anzahl Hinweise": anzahl,
        "Spätere Abmeldungen erkannt": treffer,
        "Recall": round(treffer / alle_abmeldungen * 100, 1),
        "Trefferquote": round(treffer / anzahl * 100, 1),
        "Spätere Abmeldungen übersehen": alle_abmeldungen - treffer,
    })

display(pd.DataFrame(zeilen))

,Hinweisanteil,Anzahl Hinweise,Spätere Abmeldungen erkannt,Recall,Trefferquote,Spätere Abmeldungen übersehen
0,20%,1114,389,35.1,34.9,719
1,30%,1671,525,47.4,31.4,583
2,40%,2228,639,57.7,28.7,469
3,50%,2786,747,67.4,26.8,361
4,60%,3343,840,75.8,25.1,268
5,70%,3900,923,83.3,23.7,185
6,80%,4457,999,90.2,22.4,109


In [7]:
# Referenzmodell nur mit Modul und Kursdurchlauf trainieren.
kursmodell = Pipeline([
    (
        "vorverarbeitung",
        OneHotEncoder(handle_unknown="ignore")
    ),
    (
        "klassifikation",
        LogisticRegression(max_iter=1000)
    ),
])

kursmodell.fit(
    entwick[kategoriespalten],
    entwick["spaeter_abgemeldet"]
)
kurswerte = kursmodell.predict_proba(valid[kategoriespalten])[:, 1]

# Alle drei Verfahren auf denselben Validierungsfällen vergleichen.
verfahren = {
    "Wenigste Klicks": -valid["klicks_gesamt"].to_numpy(),
    "Nur Kurskontext": kurswerte,
    "Kurs + Aktivität": modellwerte,
}

tatsaechlich = valid["spaeter_abgemeldet"].to_numpy()
alle_abmeldungen = tatsaechlich.sum()
ergebnisse = []

for name, werte in verfahren.items():
    # AP bewertet die Rangfolge über alle möglichen Hinweiszahlen.
    ap = average_precision_score(tatsaechlich, werte)

    # Höchste Werte zuerst; bei der Klickregel heißt das:
    # wegen des Minuszeichens die wenigsten Klicks zuerst.
    reihenfolge = np.argsort(-werte, kind="stable")

    for anteil in [0.20, 0.60, 0.70]:
        anzahl = round(len(valid) * anteil)
        treffer = tatsaechlich[reihenfolge[:anzahl]].sum()

        ergebnisse.append({
            "Verfahren": name,
            "Hinweisanteil": f"{anteil:.0%}",
            "AP": round(ap, 3),
            "Treffer": int(treffer),
            "Recall": round(100 * treffer / alle_abmeldungen, 1),
            "Trefferquote": round(100 * treffer / anzahl, 1),
        })

display(pd.DataFrame(ergebnisse))

,Verfahren,Hinweisanteil,AP,Treffer,Recall,Trefferquote
0,Wenigste Klicks,20%,0.239,286,25.8,25.7
1,Wenigste Klicks,60%,0.239,758,68.4,22.7
2,Wenigste Klicks,70%,0.239,865,78.1,22.2
3,Nur Kurskontext,20%,0.281,350,31.6,31.4
4,Nur Kurskontext,60%,0.281,817,73.7,24.4
5,Nur Kurskontext,70%,0.281,905,81.7,23.2
6,Kurs + Aktivität,20%,0.330,389,35.1,34.9
7,Kurs + Aktivität,60%,0.330,840,75.8,25.1
8,Kurs + Aktivität,70%,0.330,923,83.3,23.7


In [8]:
# Nur die numerischen Modellmerkmale der Entwicklungsdaten betrachten.
numerische_merkmale = [
    "anmeldetag",
    "aktive_tage_0_bis_6",
    "aktive_tage_7_bis_14",
]

# Spearman misst, ob höhere Werte eines Merkmals tendenziell
# mit höheren oder niedrigeren Werten eines anderen einhergehen.
korrelationen = entwick[numerische_merkmale].corr(method="spearman")

display(korrelationen.round(2))

,anmeldetag,aktive_tage_0_bis_6,aktive_tage_7_bis_14
anmeldetag,1.00,-0.08,-0.08
aktive_tage_0_bis_6,-0.08,1.00,0.65
aktive_tage_7_bis_14,-0.08,0.65,1.00


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Kategorien umwandeln; numerische Merkmale unverändert durchreichen.
# Ein Random Forest braucht für diese Zahlen keine Skalierung.
rf_vorverarbeitung = ColumnTransformer(
    transformers=[
        (
            "kategorien",
            OneHotEncoder(handle_unknown="ignore"),
            kategoriespalten,
        )
    ],
    remainder="passthrough",
)

rf_modell = Pipeline([
    ("vorverarbeitung", rf_vorverarbeitung),
    ("klassifikation", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
    )),
])

# Ausschließlich auf den Entwicklungsdaten trainieren.
rf_modell.fit(
    entwick[merkmale],
    entwick["spaeter_abgemeldet"]
)

# Validierungsdaten nur für die Bewertung verwenden.
rf_werte = rf_modell.predict_proba(valid[merkmale])[:, 1]
rf_ap = average_precision_score(
    valid["spaeter_abgemeldet"], rf_werte
)

print("Average Precision – Logistische Regression:", round(ap_modell, 3))
print("Average Precision – Random Forest:", round(rf_ap, 3))

# Bei gleicher Hinweiszahl und denselben Validierungsfällen vergleichen.
reihenfolge_rf = np.argsort(-rf_werte, kind="stable")
tatsaechlich = valid["spaeter_abgemeldet"].to_numpy()
alle_abmeldungen = tatsaechlich.sum()

for anteil in [0.20, 0.60, 0.70]:
    anzahl = round(len(valid) * anteil)
    treffer_rf = tatsaechlich[reihenfolge_rf[:anzahl]].sum()

    print(
        f"{anteil:.0%} Hinweise: {treffer_rf} Treffer, "
        f"Recall {treffer_rf / alle_abmeldungen:.1%}, "
        f"Trefferquote {treffer_rf / anzahl:.1%}"
    )

Average Precision – Logistische Regression: 0.33
Average Precision – Random Forest: 0.335
20% Hinweise: 390 Treffer, Recall 35.2%, Trefferquote 35.0%
60% Hinweise: 825 Treffer, Recall 74.5%, Trefferquote 24.7%
70% Hinweise: 910 Treffer, Recall 82.1%, Trefferquote 23.3%


### Zwischenfazit: Merkmalversionen und Modelle

Version 2 ersetzt die Gesamtzahl aktiver Tage durch zwei getrennte Zeitfenster. Auf der bisherigen Validierungsaufteilung zeigt sich kein deutlicher Gewinn:

| Modell | Merkmale | AP | Recall bei 20 % | Recall bei 60 % | Recall bei 70 % |
|---|---|---:|---:|---:|---:|
| Logistische Regression | Version 1 | 0,331 | 35,0 % | 75,6 % | 83,4 % |
| Logistische Regression | Version 2 | 0,330 | 35,1 % | 75,8 % | 83,3 % |
| Random Forest | Version 1 | 0,333 | 35,2 % | 74,5 % | 81,9 % |
| Random Forest | Version 2 | 0,335 | 35,2 % | 74,5 % | 82,1 % |

Die Abweichungen sind sehr klein und beruhen bisher auf nur einer Validierungsaufteilung. Version 1 liegt als `oulad_tag14_modellbasis.csv` vor, Version 2 als `oulad_tag14_modellbasis_v2.csv`. Die endgültige Modellwahl erfordert eine Prüfung über mehrere gruppenweise Aufteilungen; der Testsatz bleibt bis dahin ungenutzt.